In [25]:
# Install and import required libraries
import pandas as pd
!pip install torch imageio scikit-learn matplotlib

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import imageio
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler


In [1]:

# Upload single-cell gene expression dataset
from google.colab import files
uploaded = files.upload()

Saving single_cell.csv to single_cell.csv


In [26]:
# Load dataset and separate features (genes) and labels (grouping)
data = pd.read_csv('single_cell.csv')

In [27]:
# Remove Cell ID column and extract gene expression features
X = data.iloc[:, 1:-1].values
# Extract grouping labels (last column)
y = data.iloc[:, -1].values

print("Feature shape:", X.shape)
print("Labels shape:", y.shape)



Feature shape: (100, 199)
Labels shape: (100,)


In [28]:
# Encode categorical labels and normalize gene expression data

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# CRITICAL FIX: replace NaN and infinite values
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# Convert to tensors
X_tensor = torch.FloatTensor(X_scaled)
y_tensor = torch.LongTensor(y_encoded)

input_dim = X_scaled.shape[1]
num_classes = len(np.unique(y_encoded))

print("Input dim:", input_dim)
print("Classes:", num_classes)


Input dim: 199
Classes: 4


In [29]:
# Define neural network classifier with latent embedding layer

class Classifier(nn.Module):

    def __init__(self, input_dim, latent_dim, num_classes):
        super().__init__()

        # Encoder: compress input into latent embedding
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim)
        )

        # Classifier: predict cell group from latent embedding
        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Linear(latent_dim, num_classes)
        )

    def forward(self, x):
        latent = self.encoder(x)
        output = self.classifier(latent)
        return output, latent

# Initialize model, loss function, and optimizer
latent_dim = 2

model = Classifier(input_dim, latent_dim, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

print("Model created successfully")

Model created successfully


In [30]:
# Create folder to store visualization frames
os.makedirs("frames", exist_ok=True)

In [31]:
  # Train classifier and save latent embeddings at each epoch

epochs = 50

for epoch in range(epochs):

      optimizer.zero_grad()

      outputs, latent = model(X_tensor)

      loss = criterion(outputs, y_tensor)

      loss.backward()
      optimizer.step()

      print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

      # Extract latent
      latent_np = latent.detach().numpy()

      # Plot and save embedding visualization
      plt.figure()

      scatter = plt.scatter(
          latent_np[:, 0],
          latent_np[:, 1],
          c=y_encoded,
          cmap='tab10'
      )

      plt.title(f"Epoch {epoch+1}")
      plt.xlabel("Latent Dimension 1")
      plt.ylabel("Latent Dimension 2")

      plt.savefig(f"frames/frame_{epoch}.png")
      plt.close()

Epoch 1/50, Loss: 1.4569
Epoch 2/50, Loss: 1.4402
Epoch 3/50, Loss: 1.4229
Epoch 4/50, Loss: 1.4046
Epoch 5/50, Loss: 1.3861
Epoch 6/50, Loss: 1.3676
Epoch 7/50, Loss: 1.3497
Epoch 8/50, Loss: 1.3330
Epoch 9/50, Loss: 1.3173
Epoch 10/50, Loss: 1.3025
Epoch 11/50, Loss: 1.2886
Epoch 12/50, Loss: 1.2753
Epoch 13/50, Loss: 1.2626
Epoch 14/50, Loss: 1.2506
Epoch 15/50, Loss: 1.2392
Epoch 16/50, Loss: 1.2283
Epoch 17/50, Loss: 1.2180
Epoch 18/50, Loss: 1.2082
Epoch 19/50, Loss: 1.1988
Epoch 20/50, Loss: 1.1897
Epoch 21/50, Loss: 1.1810
Epoch 22/50, Loss: 1.1726
Epoch 23/50, Loss: 1.1644
Epoch 24/50, Loss: 1.1565
Epoch 25/50, Loss: 1.1488
Epoch 26/50, Loss: 1.1412
Epoch 27/50, Loss: 1.1338
Epoch 28/50, Loss: 1.1264
Epoch 29/50, Loss: 1.1191
Epoch 30/50, Loss: 1.1118
Epoch 31/50, Loss: 1.1046
Epoch 32/50, Loss: 1.0973
Epoch 33/50, Loss: 1.0900
Epoch 34/50, Loss: 1.0826
Epoch 35/50, Loss: 1.0753
Epoch 36/50, Loss: 1.0680
Epoch 37/50, Loss: 1.0606
Epoch 38/50, Loss: 1.0532
Epoch 39/50, Loss: 1.

In [32]:
# Create GIF showing latent embedding evolution during training

images = []

for epoch in range(epochs):
    filename = f"frames/frame_{epoch}.png"
    images.append(imageio.imread(filename))

imageio.mimsave("latent_training.gif", images, duration=0.4)

print("GIF created successfully")


/tmp/ipython-input-1676613356.py:7: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(filename))


GIF created successfully


In [33]:
# Download the generated GIF
from google.colab import files
files.download("latent_training.gif")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>